In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data = np.load("mnist.npz")

x_train, y_train = data["x_train"], data["y_train"]
x_test, y_test = data["x_test"], data["y_test"]

data.close()

In [ ]:
x_train = x_train.reshape(60000, 784).T # flatten each image
y_train = np.eye(10)[y_train].T # convert 0 to [1 0 0 0 0 0 0 0 0 0]
x_test = x_test.reshape(10000, 784).T # flatten each image
y_test = np.eye(10)[y_test].T # convert 0 to [1 0 0 0 0 0 0 0 0 0]



(10, 10000)

In [ ]:
w1 = np.random.rand(100, 784) * 2 - 1
b1 = np.random.rand(100) * 2 - 1

w2 = np.random.rand(150, 100) * 2 - 1
b2 = np.random.rand(150) * 2 - 1

w3 = np.random.rand(10, 150) * 2 - 1
b3 = np.random.rand(10) * 2 - 1

In [5]:
sigmoid = lambda x: 1 / (1 + np.exp(-x))

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=0, keepdims=True))  # Subtract max for numerical stability
    return exp_x / np.sum(exp_x, axis=0, keepdims=True)

In [6]:
def forword_passing(x):
        # x.shape = (784, batch)
        z1 = w1.dot(x) + np.repeat(b1[:, np.newaxis], x.shape[1], axis=1)
        a1 = sigmoid(z1)
        z2 = w2.dot(a1) + np.repeat(b2[:, np.newaxis], x.shape[1], axis=1)
        a2 = sigmoid(z2)
        z3 = w3.dot(a2) + np.repeat(b3[:, np.newaxis], x.shape[1], axis=1)
        a3 = softmax(z3)
        return a1, a2, a3

In [7]:
def loss_function(y_true, y_pred): # cross entropy
    return -np.mean(y_true * np.log10(y_pred) + (1 - y_true) * np.log10(1 - y_pred))


In [9]:
x_data = x_train.reshape(784, 3750, 16).transpose(1, 0, 2)
y_data = y_train.reshape(10, 3750, 16).transpose(1, 0, 2)
train_loss = []
test_loss = []

In [ ]:
for epoch in range(100):
    for x, y in zip(x_data, y_data):
        a1, a2, a3 = forword_passing(x)
        # bp
        error3 = a3 - y # (10, 16)
        dw3 = error3.dot(a2.T)

        error2 = a2 * (1 - a2) * w3.T.dot(error3)
        dw2 = error2.dot(a1.T)

        error1 = a1 * (1 - a1) * w2.T.dot(error2)
        dw1 = error1.dot(x.T)

        db3 = np.mean(error3, axis=1)
        db2 = np.mean(error2, axis=1)
        db1 = np.mean(error1, axis=1)

        w3 -= dw3 * 0.01
        w2 -= dw2 * 0.01
        w1 -= dw1 * 0.01

        b3 -= db3 * 0.01
        b2 -= db2 * 0.01
        b1 -= db1 * 0.01

    print(epoch)
    # compute loss
    _, _, pred_train = forword_passing(x_train)
    _, _, pred_test = forword_passing(x_test)
    train_loss.append(loss_function(y_train, pred_train))
    test_loss.append(loss_function(y_test, pred_test))

C:\Users\jason\AppData\Local\Temp\ipykernel_12888\3578200075.py:1: RuntimeWarning: overflow encountered in exp
  sigmoid = lambda x: 1 / (1 + np.exp(-x))


0
1
2


KeyboardInterrupt: 

In [ ]:
fig = plt.figure(1, figsize=(7, 5), dpi=100)
ax = fig.add_subplot(111)
ax.set(
    ylabel='Loss',
    xlabel='epochs',
    xlim=(0, 100),
    ylim=(0,)
)
ax.plot(
    np.arange(100),
    train_loss,
    label='Train loss',
    color='r'
)
ax.plot(
    np.arange(100),
    test_loss,
    label='Test loss',
    color='g'
)
ax.legend()
fig.show()